In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv("../Data/trainTitanic2.csv")

In [30]:
df = df[['Sex' , 'Embarked' , 'Pclass' , 'Survived']].dropna()

In [31]:
df.shape

(889, 4)

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Sex       889 non-null    object
 1   Embarked  889 non-null    object
 2   Pclass    889 non-null    int64 
 3   Survived  889 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 34.7+ KB


In [33]:
from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder()
X = encoder.fit_transform(
    df[['Sex' , 'Embarked' , 'Pclass']]
)
y = df['Survived'].to_numpy()

In [34]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train ,y_test = train_test_split(X ,y , test_size=0.2 , stratify=y)

In [35]:
class CatNB:
    
    def __init__(self , alpha = 1): # for smoothing 
        self.alpha = alpha
        self.classes = None
        self.priors = None
        self.category_probabilities = []
        self.n_categories = None
        
        
    def fit(self , X , y):
        self.classes = np.unique(y)
        number_of_classes = len(self.classes)
        number_of_features = X.shape[1]
        
        self.n_categories = [int(np.max(X[:, j])) + 1 for j in range(number_of_features)]
        self.priors = np.zeros(number_of_classes)
        
        for i , c in enumerate(self.classes):
            X_Class = X[y==c]
            self.priors[i] = len(X_Class)/len(X)
            
        self.categorical_probabilities = []
        for j in range(number_of_features):
            k = self.n_categories[j]
            
            features_probab = np.zeros((number_of_classes , k))
            for i, c in enumerate(self.classes):

                X_c = X[y == c]

                counts = np.bincount( X_c[:, j].astype(int),minlength=k)

                features_probab[i] = (counts + self.alpha) / (len(X_c) + self.alpha * k)

            self.category_probabilities.append(
                features_probab
            )
            
        return self
    
    def predict(self, X):

        predictions = []
        for x in X:
            class_probabilities = []

            for i, c in enumerate(self.classes):
                probability = self.priors[i]
                
                for j in range(X.shape[1]):
                    category = int(x[j])
                    probability *= ( self.category_probabilities[j][i, category] )
                    
                class_probabilities.append(probability)
                
            best_class = self.classes[
                np.argmax(class_probabilities)
            ]
            predictions.append(best_class)

        return np.array(predictions)

In [36]:
my_nb = CatNB(alpha=1.0)
my_nb.fit(X_train, y_train)
y_pred_scratch = my_nb.predict(X_test)

In [37]:
from sklearn.naive_bayes import CategoricalNB
sk_nb = CategoricalNB(alpha=1.0)
sk_nb.fit(X_train, y_train)
y_pred_sklearn = sk_nb.predict(X_test)

In [38]:
from sklearn.metrics import accuracy_score

print(
    "Scratch:",
    accuracy_score(y_test, y_pred_scratch)
)

print(
    "Sklearn:",
    accuracy_score(y_test, y_pred_sklearn)
)

Scratch: 0.7471910112359551
Sklearn: 0.7471910112359551
